# Delta Hedging and the Gamma-Theta Tradeoff

In [ ]:
import sys
sys.path.insert(0, '..')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../results/figures', exist_ok=True)
os.makedirs('../results/tables', exist_ok=True)

from src.delta_hedge import run_delta_hedge, single_path_detail, gamma_pnl_decomposition
from src.pricing import BlackScholes
from src import config

print('All imports successful.')

# 1. Parameters

In [ ]:
S0 = 100      # Initial spot
K = 100       # Strike (ATM)
T = 0.25      # 3 months
r = 0.05      # Risk-free rate
q = 0         # No dividends

n_sims = 10_000
n_rebalances = 252  # Daily rebalancing

# Option premium for reference
bs_ref = BlackScholes(S0, K, T, r, q, 0.20)
print(f'Reference call premium (sigma=20%): ${float(bs_ref.call_price()):.4f}')
print(f'Parameters: S0={S0}, K={K}, T={T}, r={r}, q={q}')
print(f'Simulations: {n_sims:,}, Rebalances: {n_rebalances}')

# 2. Perfect Hedge

In [ ]:
perfect = run_delta_hedge(
    S0=S0, K=K, T=T, r=r, q=q,
    sigma_true=0.20, sigma_hedge=0.20,
    n_rebalances=n_rebalances, n_simulations=n_sims,
    option_type='call', seed=42
)

print('Perfect Hedge (sigma_true = sigma_hedge = 20%)')
print('=' * 55)
print(f'Mean P&L:        ${perfect["mean_pnl"]:.4f}')
print(f'Std P&L:         ${perfect["std_pnl"]:.4f}')
print(f'Median P&L:      ${perfect["median_pnl"]:.4f}')
print(f'Sharpe:          {perfect["sharpe"]:.4f}')
print(f'% Profitable:    {perfect["pct_profitable"]:.1f}%')
print(f'5th percentile:  ${perfect["percentiles"]["p5"]:.4f}')
print(f'95th percentile: ${perfect["percentiles"]["p95"]:.4f}')

# 3. Vol Underestimate

In [ ]:
underestimate = run_delta_hedge(
    S0=S0, K=K, T=T, r=r, q=q,
    sigma_true=0.25, sigma_hedge=0.20,
    n_rebalances=n_rebalances, n_simulations=n_sims,
    option_type='call', seed=42
)

print('Vol Underestimate (sigma_true=25%, sigma_hedge=20%)')
print('=' * 55)
print(f'Mean P&L:        ${underestimate["mean_pnl"]:.4f}')
print(f'Std P&L:         ${underestimate["std_pnl"]:.4f}')
print(f'% Profitable:    {underestimate["pct_profitable"]:.1f}%')

# 4. Vol Overestimate

In [ ]:
overestimate = run_delta_hedge(
    S0=S0, K=K, T=T, r=r, q=q,
    sigma_true=0.15, sigma_hedge=0.20,
    n_rebalances=n_rebalances, n_simulations=n_sims,
    option_type='call', seed=42
)

print('Vol Overestimate (sigma_true=15%, sigma_hedge=20%)')
print('=' * 55)
print(f'Mean P&L:        ${overestimate["mean_pnl"]:.4f}')
print(f'Std P&L:         ${overestimate["std_pnl"]:.4f}')
print(f'% Profitable:    {overestimate["pct_profitable"]:.1f}%')

# 5. Combined P&L Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

scenarios = [
    ('Perfect (20/20)', perfect, 'steelblue'),
    ('Underestimate (25/20)', underestimate, 'crimson'),
    ('Overestimate (15/20)', overestimate, 'forestgreen'),
]

for label, result, color in scenarios:
    pnl = result['final_pnl']
    ax.hist(pnl, bins=80, alpha=0.45, color=color, edgecolor=color,
            linewidth=0.5, density=True, label=f'{label} (mean=${result["mean_pnl"]:.3f})')
    ax.axvline(x=result['mean_pnl'], color=color, linestyle='--', linewidth=1.5)

ax.axvline(x=0, color='black', linewidth=1)
ax.set_xlabel('Hedge P&L ($)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Delta-Hedge P&L Distribution: Vol Misspecification',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('../results/figures/hedge_pnl_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/hedge_pnl_distribution.png')

In [ ]:
# Summary table
hedging_rows = []
for label, result, _ in scenarios:
    hedging_rows.append({
        'Scenario': label,
        'Mean PnL': round(result['mean_pnl'], 4),
        'Std PnL': round(result['std_pnl'], 4),
        'Median PnL': round(result['median_pnl'], 4),
        'Sharpe': round(result['sharpe'], 4),
        'Pct Profitable': round(result['pct_profitable'], 1),
        'P5': round(result['percentiles']['p5'], 4),
        'P95': round(result['percentiles']['p95'], 4),
    })

hedging_df = pd.DataFrame(hedging_rows)
print(hedging_df.to_string(index=False))

hedging_df.to_csv('../results/tables/hedging_results.csv', index=False)
print('\nSaved: results/tables/hedging_results.csv')

# 6. Single Path Detail

In [ ]:
path_detail = single_path_detail(
    S0=S0, K=K, T=T, r=r, q=q,
    sigma_true=0.15, sigma_hedge=0.20,
    n_rebalances=n_rebalances,
    option_type='call', seed=42
)

time_grid = np.linspace(0, T, n_rebalances + 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Stock price path
ax = axes[0, 0]
ax.plot(time_grid, path_detail['S_path'], color='steelblue', linewidth=1.5)
ax.axhline(y=K, color='crimson', linestyle='--', linewidth=1, label=f'K={K}')
ax.set_xlabel('Time (years)', fontsize=10)
ax.set_ylabel('Stock Price ($)', fontsize=10)
ax.set_title('Stock Price Path', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)

# Panel 2: Delta (hedge ratio)
ax = axes[0, 1]
ax.plot(time_grid, path_detail['delta_path'], color='darkorange', linewidth=1.5)
ax.set_xlabel('Time (years)', fontsize=10)
ax.set_ylabel('Delta (shares held)', fontsize=10)
ax.set_title('Hedge Ratio (Delta)', fontsize=12, fontweight='bold')

# Panel 3: Portfolio value vs option value
ax = axes[1, 0]
ax.plot(time_grid, path_detail['portfolio_value'], color='steelblue', linewidth=1.5,
        label='Hedge Portfolio')
ax.plot(time_grid, path_detail['option_value'], color='crimson', linewidth=1.5,
        linestyle='--', label='Option Value')
ax.set_xlabel('Time (years)', fontsize=10)
ax.set_ylabel('Value ($)', fontsize=10)
ax.set_title('Portfolio vs Option Value', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)

# Panel 4: Cumulative hedge P&L
ax = axes[1, 1]
ax.plot(time_grid, path_detail['hedge_pnl'], color='forestgreen', linewidth=1.5)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Time (years)', fontsize=10)
ax.set_ylabel('Hedge P&L ($)', fontsize=10)
ax.set_title(f'Hedge P&L (final = ${path_detail["hedge_pnl"][-1]:.4f})', fontsize=12, fontweight='bold')
ax.fill_between(time_grid, 0, path_detail['hedge_pnl'],
                where=path_detail['hedge_pnl'] >= 0, alpha=0.2, color='forestgreen')
ax.fill_between(time_grid, 0, path_detail['hedge_pnl'],
                where=path_detail['hedge_pnl'] < 0, alpha=0.2, color='crimson')

plt.suptitle('Single Path Delta-Hedge Detail (sigma_true=15%, sigma_hedge=20%)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/single_path_detail.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/single_path_detail.png')

# 7. Gamma P&L Scatter

In [ ]:
dS = np.diff(path_detail['S_path'])
gamma_pnl = path_detail['gamma_pnl_per_step']

fig, ax = plt.subplots(figsize=(10, 7))

scatter = ax.scatter(dS, gamma_pnl, c=np.arange(len(dS)),
                     cmap='viridis', alpha=0.6, s=15, edgecolors='none')

# Quadratic fit overlay
dS_sorted = np.linspace(dS.min(), dS.max(), 100)
# Average gamma over the path
avg_gamma = np.mean([float(BlackScholes(s, K, max(T - i * T / n_rebalances, 0.001), r, q, 0.20).gamma())
                     for i, s in enumerate(path_detail['S_path'][:-1])])
ax.plot(dS_sorted, 0.5 * avg_gamma * dS_sorted**2, 'r--', linewidth=2,
        label=f'0.5 * avg_gamma * dS^2')

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Time Step', fontsize=10)

ax.set_xlabel('Price Change dS ($)', fontsize=12)
ax.set_ylabel('Gamma P&L ($)', fontsize=12)
ax.set_title('Per-Period Gamma P&L vs Price Move', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../results/figures/gamma_pnl_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/gamma_pnl_scatter.png')

# 8. Gamma-Theta Decomposition

In [ ]:
decomp = gamma_pnl_decomposition(path_detail)

cum_actual = np.cumsum(decomp['actual_pnl'])
cum_gamma = np.cumsum(decomp['gamma_component'])
cum_theta = np.cumsum(decomp['theta_component'])
cum_residual = np.cumsum(decomp['residual'])

time_steps = time_grid[1:]  # n_rebalances points

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Cumulative decomposition
ax = axes[0]
ax.plot(time_steps, cum_actual, color='black', linewidth=2, label='Actual P&L')
ax.plot(time_steps, cum_gamma, color='steelblue', linewidth=1.5, label='Gamma Component')
ax.plot(time_steps, cum_theta, color='crimson', linewidth=1.5, label='Theta Component')
ax.plot(time_steps, cum_residual, color='gray', linewidth=1, linestyle=':', label='Residual')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.3)
ax.set_xlabel('Time (years)', fontsize=11)
ax.set_ylabel('Cumulative P&L ($)', fontsize=11)
ax.set_title('Cumulative P&L Decomposition', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)

# Right: Per-period components
ax = axes[1]
ax.bar(time_steps, decomp['gamma_component'], width=T/n_rebalances * 0.8,
       color='steelblue', alpha=0.6, label='Gamma P&L')
ax.bar(time_steps, decomp['theta_component'], width=T/n_rebalances * 0.8,
       color='crimson', alpha=0.6, label='Theta P&L')
ax.set_xlabel('Time (years)', fontsize=11)
ax.set_ylabel('Per-Period P&L ($)', fontsize=11)
ax.set_title('Per-Period Gamma vs Theta', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)

plt.suptitle('Gamma-Theta P&L Decomposition', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/gamma_theta_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Total gamma P&L:  ${cum_gamma[-1]:.4f}')
print(f'Total theta P&L:  ${cum_theta[-1]:.4f}')
print(f'Total residual:   ${cum_residual[-1]:.4f}')
print(f'Actual total:     ${cum_actual[-1]:.4f}')
print('\nSaved: results/figures/gamma_theta_decomposition.png')

# 9. Rebalancing Frequency

In [ ]:
rebal_freqs = [12, 52, 252, 1260]
rebal_labels = ['Monthly (12)', 'Weekly (52)', 'Daily (252)', 'Hourly (1260)']
n_sims_rebal = 5000

rebal_results = []
for freq, label in zip(rebal_freqs, rebal_labels):
    res = run_delta_hedge(
        S0=S0, K=K, T=T, r=r, q=q,
        sigma_true=0.20, sigma_hedge=0.20,
        n_rebalances=freq, n_simulations=n_sims_rebal,
        option_type='call', seed=42
    )
    rebal_results.append({
        'Frequency': label,
        'N_rebalances': freq,
        'Mean PnL': round(res['mean_pnl'], 4),
        'Std PnL': round(res['std_pnl'], 4),
        'Pct Profitable': round(res['pct_profitable'], 1),
        'P5': round(res['percentiles']['p5'], 4),
        'P95': round(res['percentiles']['p95'], 4),
        '_final_pnl': res['final_pnl'],
    })
    print(f'{label}: Mean=${res["mean_pnl"]:.4f}, Std=${res["std_pnl"]:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Overlaid histograms
ax = axes[0]
colors_rebal = ['#c44e52', '#4c72b0', '#55a868', '#8172b2']
for res, color in zip(rebal_results, colors_rebal):
    ax.hist(res['_final_pnl'], bins=60, alpha=0.4, color=color,
            density=True, label=res['Frequency'])

ax.axvline(x=0, color='black', linewidth=1)
ax.set_xlabel('Hedge P&L ($)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('P&L by Rebalancing Frequency', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)

# Right: Std PnL vs frequency
ax = axes[1]
stds = [r['Std PnL'] for r in rebal_results]
freqs = [r['N_rebalances'] for r in rebal_results]
ax.plot(freqs, stds, 'o-', color='darkorange', linewidth=2, markersize=8)

# Theoretical: discrete hedging error scales as O(1/sqrt(N))
ref_std = stds[0] * np.sqrt(freqs[0])
freq_fine = np.linspace(freqs[0], freqs[-1], 100)
ax.plot(freq_fine, ref_std / np.sqrt(freq_fine), '--', color='gray',
        label=r'$O(1/\sqrt{N})$ reference')

ax.set_xlabel('Number of Rebalances', fontsize=11)
ax.set_ylabel('Std of Hedge P&L ($)', fontsize=11)
ax.set_title('Hedge Error vs Rebalancing Frequency', fontsize=12, fontweight='bold')
ax.set_xscale('log')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('../results/figures/rebalancing_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/rebalancing_tradeoff.png')

# Save table
rebal_table = pd.DataFrame([{k: v for k, v in r.items() if k != '_final_pnl'}
                             for r in rebal_results])
rebal_table.to_csv('../results/tables/rebalancing_comparison.csv', index=False)
print('Saved: results/tables/rebalancing_comparison.csv')
print(rebal_table.to_string(index=False))

# 10. Transaction Cost Impact

In [ ]:
tc_levels = [0, 1, 5, 10]  # basis points
n_sims_tc = 5000

tc_results = []
for tc in tc_levels:
    res = run_delta_hedge(
        S0=S0, K=K, T=T, r=r, q=q,
        sigma_true=0.20, sigma_hedge=0.20,
        n_rebalances=252, n_simulations=n_sims_tc,
        option_type='call', transaction_cost_bps=tc, seed=42
    )
    tc_results.append({
        'TC (bps)': tc,
        'Mean PnL': round(res['mean_pnl'], 4),
        'Std PnL': round(res['std_pnl'], 4),
        'Pct Profitable': round(res['pct_profitable'], 1),
        'Avg TC Cost': round(float(np.mean(res['total_transaction_costs'])), 4),
        '_final_pnl': res['final_pnl'],
    })
    print(f'TC={tc}bps: Mean=${res["mean_pnl"]:.4f}, Avg TC=${float(np.mean(res["total_transaction_costs"])):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Overlaid histograms
ax = axes[0]
colors_tc = ['#55a868', '#4c72b0', '#c44e52', '#8172b2']
for res, color in zip(tc_results, colors_tc):
    ax.hist(res['_final_pnl'], bins=60, alpha=0.4, color=color,
            density=True, label=f'{res["TC (bps)"]} bps')

ax.axvline(x=0, color='black', linewidth=1)
ax.set_xlabel('Hedge P&L ($)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('P&L by Transaction Cost Level', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)

# Right: Mean PnL and avg TC cost
ax = axes[1]
tc_vals = [r['TC (bps)'] for r in tc_results]
mean_pnls = [r['Mean PnL'] for r in tc_results]
avg_tcs = [r['Avg TC Cost'] for r in tc_results]

ax.bar(np.arange(len(tc_vals)) - 0.15, mean_pnls, 0.3, color='steelblue',
       label='Mean P&L', edgecolor='black', linewidth=0.5)
ax.bar(np.arange(len(tc_vals)) + 0.15, [-tc for tc in avg_tcs], 0.3, color='crimson',
       label='Avg TC Cost (negative)', edgecolor='black', linewidth=0.5)
ax.set_xticks(range(len(tc_vals)))
ax.set_xticklabels([f'{tc} bps' for tc in tc_vals])
ax.set_xlabel('Transaction Cost', fontsize=11)
ax.set_ylabel('P&L ($)', fontsize=11)
ax.set_title('Impact of Transaction Costs', fontsize=12, fontweight='bold')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('../results/figures/transaction_cost_impact.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/transaction_cost_impact.png')

# Save table
tc_table = pd.DataFrame([{k: v for k, v in r.items() if k != '_final_pnl'}
                          for r in tc_results])
tc_table.to_csv('../results/tables/transaction_cost_comparison.csv', index=False)
print('Saved: results/tables/transaction_cost_comparison.csv')
print(tc_table.to_string(index=False))

## Key Insight

The **delta-hedged P&L of a short option position** is fundamentally driven by the difference between the implied volatility (used for hedging) and the realized volatility:

$$\text{P\&L} \approx \frac{1}{2} \int_0^T \Gamma_t S_t^2 \left( \sigma_{\text{hedge}}^2 - \sigma_{\text{realized}}^2 \right) dt$$

- If $\sigma_{\text{hedge}} > \sigma_{\text{realized}}$: the seller collected excess premium and profits
- If $\sigma_{\text{hedge}} < \sigma_{\text{realized}}$: the seller undercharged and loses money
- The P&L is path-dependent and gamma-weighted

## Interview Talking Points

1. **Gamma scalping**: A long gamma position profits from realized moves (each $\Delta S$ generates $\frac{1}{2}\Gamma (\Delta S)^2$ P&L) but bleeds theta every day. Break-even is when realized vol equals implied vol.

2. **Discrete vs continuous hedging**: Continuous delta hedging would perfectly replicate the option. Discrete hedging introduces "slippage" that scales as $O(1/\sqrt{N})$ in standard deviation.

3. **Transaction costs**: Create a tension with rebalancing frequency. More frequent hedging reduces discretization error but increases transaction costs. The optimal frequency depends on the cost-per-trade and gamma exposure.

4. **Vol of vol**: The P&L distribution is wider when vol itself is uncertain. In practice, realized vol is stochastic, making the hedge P&L more dispersed than the GBM model suggests.

5. **P&L attribution**: Decomposing hedge P&L into gamma, theta, and residual components is essential for understanding whether profits come from vol mismatch, gamma scalping, or other sources.

In [ ]:
saved_files = [
    'results/figures/hedge_pnl_distribution.png',
    'results/tables/hedging_results.csv',
    'results/figures/single_path_detail.png',
    'results/figures/gamma_pnl_scatter.png',
    'results/figures/gamma_theta_decomposition.png',
    'results/figures/rebalancing_tradeoff.png',
    'results/tables/rebalancing_comparison.csv',
    'results/figures/transaction_cost_impact.png',
    'results/tables/transaction_cost_comparison.csv',
]

print('Files saved by this notebook:')
print('=' * 50)
for f in saved_files:
    print(f'  {f}')